# 04 — Parameter Sweep
Grid search + Optuna over deltas, exits, and signal thresholds.
**Warning:** everything here is in-sample — notebook 05 is where claims get
tested out-of-sample.

In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

In [ ]:
from lab.backtest import StrategyConfig
from lab.experiments import grid_sweep, optuna_search
from lab.report import param_heatmap

base = StrategyConfig.from_yaml("../configs/short_put_45dte.yaml").replace(
    start="2016-01-01", end="2023-12-31")

In [ ]:
grid = {
    "leg1_delta.target": [0.10, 0.16, 0.30],
    "take_profit":       [0.25, 0.5, None],
    "stop_loss":         [-2.0, None],
}
sweep = grid_sweep(base, grid, tag="sweep04")
sweep.head(10).round(3)

In [ ]:
param_heatmap(sweep, x="leg1_delta.target", y="take_profit", metric="sharpe_ratio");

In [ ]:
# Parameter stability: a real edge degrades smoothly around the optimum.
param_heatmap(sweep, x="leg1_delta.target", y="take_profit", metric="max_drawdown");

## Optuna for the bigger space (adds signal threshold)

In [ ]:
def space(trial):
    return {
        "leg1_delta.target": trial.suggest_float("delta", 0.08, 0.35),
        "take_profit":       trial.suggest_float("take_profit", 0.2, 0.8),
        "entry_filter":      f"vix_rank > {trial.suggest_float('vix_rank_min', 0.0, 0.8):.2f}",
    }

study = optuna_search(base, space, n_trials=40, tag="optuna04")
print("best:", study.best_params, "->", round(study.best_value, 3))
study.trials_dataframe()[["number", "value", "params_delta", "params_take_profit", "params_vix_rank_min"]].tail(10)